<a href="https://colab.research.google.com/github/Dhruvipatel6527/My-Colab-Project/blob/main/copy_of_DHRUVI_FINAL_PDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# =============================================================================
#        RIDE-SHARING ANALYTICS AND FARE PREDICTION USING PYTHON
#                 GOOGLE COLAB SINGLE-CELL VERSION
#                    CIPAT - COMPLETE SINGLE CELL
# =============================================================================
#
# Covers:
# 1. Python Core Concepts
# 2. Data Structures
# 3. Python Libraries
# 4. Custom Modules
# 5. Exception Handling
# 6. Web Scraping
# 7. File Handling
# 8. CRUD Operations
# 9. NumPy
# 10. SciPy
# 11. Pandas
# 12. Data Preprocessing
# 13. Data Visualization
# 14. Statistical Analysis
# 15. Probability
# 16. Sampling & Inference
# 17. Hypothesis Testing
# 18. Machine Learning
# 19. Recommendation System
#
# Dataset:
# uber_trips_dataset_50k.csv
#
# =============================================================================


# =============================================================================
# 1. PYTHON LIBRARIES
# =============================================================================

import os
import sys
import zipfile
import warnings
import requests
import importlib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

print("=" * 80)
print("       RIDE-SHARING ANALYTICS AND FARE PREDICTION")
print("=" * 80)
print("All required Python libraries imported successfully.")


# =============================================================================
# 2. FILE HANDLING - UPLOAD DATASET
# =============================================================================

print("\n" + "=" * 80)
print("2. FILE HANDLING - DATASET UPLOAD")
print("=" * 80)

try:

    from google.colab import files

    print("Please upload your Uber Trips 50K CSV or ZIP file.")

    uploaded = files.upload()

    print("File uploaded successfully.")

except Exception as e:

    print("Upload error:", e)


# =============================================================================
# 3. FIND AND LOAD THE DATASET
# =============================================================================

print("\n" + "=" * 80)
print("3. LOADING DATASET")
print("=" * 80)

try:

    content_path = "/content"

    zip_files = [
        f for f in os.listdir(content_path)
        if f.lower().endswith(".zip")
    ]

    csv_files = [
        f for f in os.listdir(content_path)
        if f.lower().endswith(".csv")
    ]

    # ---------------------------------------------------------
    # CASE 1: ZIP FILE
    # ---------------------------------------------------------

    if len(zip_files) > 0:

        zip_path = os.path.join(
            content_path,
            zip_files[0]
        )

        print("ZIP file found:", zip_files[0])

        extraction_folder = "/content/uber_dataset"

        os.makedirs(
            extraction_folder,
            exist_ok=True
        )

        with zipfile.ZipFile(
            zip_path,
            "r"
        ) as zip_ref:

            zip_ref.extractall(
                extraction_folder
            )

        csv_path = None

        for root, directories, filenames in os.walk(
            extraction_folder
        ):

            for filename in filenames:

                if filename.lower().endswith(".csv"):

                    csv_path = os.path.join(
                        root,
                        filename
                    )

                    break

            if csv_path is not None:
                break

        if csv_path is None:

            raise FileNotFoundError(
                "No CSV file found inside the ZIP."
            )

    # ---------------------------------------------------------
    # CASE 2: CSV FILE
    # ---------------------------------------------------------

    elif len(csv_files) > 0:

        expected = "/content/uber_trips_dataset_50k.csv"

        if os.path.exists(expected):

            csv_path = expected

        else:

            csv_path = os.path.join(
                content_path,
                csv_files[0]
            )

        print("CSV file found:", csv_path)

    else:

        raise FileNotFoundError(
            "No CSV or ZIP file found. Please upload your dataset."
        )

    # Load CSV

    df = pd.read_csv(
        csv_path
    )

    print("\nDataset loaded successfully.")
    print("File:", csv_path)

except Exception as e:

    print("\nERROR WHILE LOADING DATASET:")
    print(e)

    raise


# =============================================================================
# 4. PANDAS - BASIC DATASET INFORMATION
# =============================================================================

print("\n" + "=" * 80)
print("4. PANDAS - DATASET OVERVIEW")
print("=" * 80)

print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")

for number, column in enumerate(
    df.columns,
    start=1
):

    print(
        number,
        ".",
        column
    )

print("\nFirst 5 records:")

display(
    df.head()
)

print("\nLast 5 records:")

display(
    df.tail()
)

print("\nDataset information:")

df.info()


# =============================================================================
# STATISTICAL HELPER FUNCTIONS
# =============================================================================

def manual_mean(values):

    values = list(values)

    if len(values) == 0:
        return 0

    total = 0

    for value in values:
        total += value

    return total / len(values)


def manual_median(values):

    values = sorted(list(values))

    n = len(values)

    if n == 0:
        return 0

    middle = n // 2

    if n % 2 == 1:
        return values[middle]

    return (
        values[middle - 1]
        +
        values[middle]
    ) / 2


def manual_mode(values):

    frequency = {}

    for value in values:

        frequency[value] = (
            frequency.get(value, 0) + 1
        )

    if len(frequency) == 0:
        return None

    maximum_frequency = max(
        frequency.values()
    )

    for value, count in frequency.items():

        if count == maximum_frequency:
            return value


def manual_sample_variance(values):

    values = list(values)

    n = len(values)

    if n < 2:
        return 0

    mean_value = manual_mean(values)

    squared_deviation_sum = 0

    for value in values:

        squared_deviation_sum += (
            value - mean_value
        ) ** 2

    return (
        squared_deviation_sum
        /
        (n - 1)
    )


def manual_sample_std(values):

    return np.sqrt(
        manual_sample_variance(values)
    )


def manual_correlation(x_values, y_values):

    x_values = list(x_values)
    y_values = list(y_values)

    n = len(x_values)

    if n == 0 or n != len(y_values):
        return 0

    x_mean = manual_mean(x_values)
    y_mean = manual_mean(y_values)

    numerator = 0
    x_squared_sum = 0
    y_squared_sum = 0

    for i in range(n):

        x_difference = (
            x_values[i] - x_mean
        )

        y_difference = (
            y_values[i] - y_mean
        )

        numerator += (
            x_difference
            *
            y_difference
        )

        x_squared_sum += (
            x_difference ** 2
        )

        y_squared_sum += (
            y_difference ** 2
        )

    denominator = np.sqrt(
        x_squared_sum
        *
        y_squared_sum
    )

    if denominator == 0:
        return 0

    return numerator / denominator


# =============================================================================
# 5. PYTHON CORE CONCEPTS
# =============================================================================

print("\n" + "=" * 80)
print("5. PYTHON CORE CONCEPTS")
print("=" * 80)

total_rides = len(df)

fare_values_initial = df[
    "fare_amount"
].dropna()

average_fare = manual_mean(
    fare_values_initial.tolist()
)

distance_values_initial = df[
    "distance_km"
].dropna()

average_distance = manual_mean(
    distance_values_initial.tolist()
)


def calculate_percentage(
    part,
    total
):

    if total == 0:

        return 0

    return (
        part / total
    ) * 100


if average_fare > 0:

    fare_message = (
        "Average fare is greater than zero."
    )

else:

    fare_message = (
        "Average fare is zero or invalid."
    )


print(
    "Total rides:",
    total_rides
)

print(
    "Average fare:",
    round(
        average_fare,
        2
    )
)

print(
    "Average distance:",
    round(
        average_distance,
        2
    )
)

print(
    fare_message
)


print("\nFirst 5 cities using a loop:")

for city in df[
    "city"
].dropna().unique()[:5]:

    print(
        "-",
        city
    )


# =============================================================================
# 6. DATA STRUCTURES
# =============================================================================

print("\n" + "=" * 80)
print("6. DATA STRUCTURES")
print("=" * 80)

cities_list = (
    df["city"]
    .dropna()
    .unique()
    .tolist()
)

ride_categories = (
    "Short",
    "Medium",
    "Long"
)

cities_set = set(
    cities_list
)

ride_summary = {

    "Total Rides":
        len(df),

    "Cities":
        len(cities_set),

    "Average Fare":
        round(
            df["fare_amount"].sum()
            /
            df["fare_amount"].count(),
            2
        ),

    "Average Distance":
        round(
            df["distance_km"].sum()
            /
            df["distance_km"].count(),
            2
        )
}

print("\nLIST:")
print(cities_list)

print("\nTUPLE:")
print(ride_categories)

print("\nSET:")
print(cities_set)

print("\nDICTIONARY:")
print(ride_summary)


# =============================================================================
# 7. CUSTOM MODULE
# =============================================================================

print("\n" + "=" * 80)
print("7. CUSTOM MODULE")
print("=" * 80)

module_code = """

def calculate_average(values):

    if len(values) == 0:
        return 0

    return sum(values) / len(values)


def calculate_percentage(part, total):

    if total == 0:
        return 0

    return (part / total) * 100

"""

module_path = "/content/ride_functions.py"

with open(
    module_path,
    "w"
) as file:

    file.write(
        module_code
    )

try:

    importlib.invalidate_caches()

    import ride_functions

    print(
        "Custom module file created:"
    )

    print(
        module_path
    )

    print(
        "\nCustom module demonstrates reusable functions."
    )

except Exception as e:

    print(
        "Custom module error:",
        e
    )


module_average = ride_functions.calculate_average(
    df["fare_amount"].dropna().tolist()
)

module_percentage = ride_functions.calculate_percentage(
    len(df),
    len(df)
)

print(
    "Average fare calculated using custom module:",
    round(module_average, 2)
)

print(
    "Custom module percentage example:",
    round(module_percentage, 2),
    "%"
)


# =============================================================================
# 8. EXCEPTION HANDLING
# =============================================================================

print("\n" + "=" * 80)
print("8. EXCEPTION HANDLING")
print("=" * 80)

try:

    fare_values = df[
        "fare_amount"
    ].dropna()

    test_value = (
        fare_values.sum()
        /
        fare_values.count()
    )

    print(
        "Fare calculation successful:",
        round(
            test_value,
            2
        )
    )

except KeyError:

    print(
        "Required fare column was not found."
    )

except Exception as e:

    print(
        "Unexpected error:",
        e
    )


# =============================================================================
# 9. DATA PREPROCESSING
# =============================================================================

print("\n" + "=" * 80)
print("9. DATA PREPROCESSING")
print("=" * 80)

df = df.copy()

before_duplicates = len(df)

df = df.drop_duplicates()

after_duplicates = len(df)

print(
    "Duplicate rows removed:",
    before_duplicates - after_duplicates
)


df["pickup_time"] = pd.to_datetime(
    df["pickup_time"],
    errors="coerce"
)

df["drop_time"] = pd.to_datetime(
    df["drop_time"],
    errors="coerce"
)


numeric_columns = [
    "distance_km",
    "fare_amount",
    "pickup_lat",
    "pickup_lng",
    "drop_lat",
    "drop_lng"
]

for column in numeric_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


df = df[
    df["distance_km"] > 0
]

df = df[
    df["fare_amount"] >= 0
]


df["duration_min"] = (
    (
        df["drop_time"]
        -
        df["pickup_time"]
    )
    .dt.total_seconds()
    / 60
)


df = df[
    df["duration_min"] > 0
]

df = df[
    df["duration_min"] < 300
]


categorical_columns = [
    "city",
    "status",
    "payment_method"
]

for column in categorical_columns:

    if column in df.columns:

        df[column] = df[column].fillna(
            "Unknown"
        )


# =============================================================================
# 10. CREATE ANALYTICAL COLUMNS
# =============================================================================

print("\n" + "=" * 80)
print("10. DATA TRANSFORMATION")
print("=" * 80)

df["hour"] = (
    df["pickup_time"]
    .dt.hour
)

df["day_of_week"] = (
    df["pickup_time"]
    .dt.day_name()
)

df["is_weekend"] = (
    df["pickup_time"]
    .dt.dayofweek >= 5
)

df["peak_hour"] = (
    (
        (df["hour"] >= 7) &
        (df["hour"] <= 9)
    )
    |
    (
        (df["hour"] >= 17) &
        (df["hour"] <= 20)
    )
)

df["peak_numeric"] = (
    df["peak_hour"]
    .astype(int)
)


df["distance_category"] = pd.cut(
    df["distance_km"],
    bins=[
        -np.inf,
        5,
        10,
        np.inf
    ],
    labels=[
        "Short",
        "Medium",
        "Long"
    ]
)


df["fare_per_km"] = (
    df["fare_amount"]
    /
    df["distance_km"]
)


df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

df.dropna(
    subset=[
        "fare_amount",
        "distance_km",
        "duration_min"
    ],
    inplace=True
)


print(
    "New analytical columns created:"
)

new_columns = [
    "duration_min",
    "hour",
    "day_of_week",
    "is_weekend",
    "peak_hour",
    "peak_numeric",
    "distance_category",
    "fare_per_km"
]

print(
    new_columns
)

print("\nUpdated dataset preview:")

display(
    df.head()
)


# =============================================================================
# 11. NUMPY
# =============================================================================

print("\n" + "=" * 80)
print("11. NUMPY")
print("=" * 80)

fare_array = np.array(
    df["fare_amount"]
)

distance_array = np.array(
    df["distance_km"]
)


numpy_mean_fare = (
    np.sum(fare_array)
    /
    len(fare_array)
)


sorted_fare_array = np.sort(
    fare_array
)

n_fare = len(
    sorted_fare_array
)

if n_fare % 2 == 1:

    numpy_median_fare = (
        sorted_fare_array[n_fare // 2]
    )

else:

    numpy_median_fare = (
        sorted_fare_array[n_fare // 2 - 1]
        +
        sorted_fare_array[n_fare // 2]
    ) / 2


numpy_min_fare = sorted_fare_array[0]

numpy_max_fare = sorted_fare_array[-1]


numpy_mean_for_std = (
    np.sum(fare_array)
    /
    len(fare_array)
)

numpy_variance_population = (
    np.sum(
        (
            fare_array
            -
            numpy_mean_for_std
        ) ** 2
    )
    /
    len(fare_array)
)

numpy_std_population = np.sqrt(
    numpy_variance_population
)


print(
    "NumPy Mean Fare:",
    round(
        numpy_mean_fare,
        2
    )
)

print(
    "NumPy Median Fare:",
    round(
        numpy_median_fare,
        2
    )
)

print(
    "NumPy Minimum Fare:",
    round(
        numpy_min_fare,
        2
    )
)

print(
    "NumPy Maximum Fare:",
    round(
        numpy_max_fare,
        2
    )
)

print(
    "NumPy Standard Deviation:",
    round(
        numpy_std_population,
        2
    )
)


# =============================================================================
# 12. STATISTICAL ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("12. STATISTICAL ANALYSIS")
print("=" * 80)

fare = df[
    "fare_amount"
].dropna()

distance = df[
    "distance_km"
].dropna()

duration = df[
    "duration_min"
].dropna()


fare_values = fare.tolist()

fare_n = len(fare_values)

fare_mean = manual_mean(
    fare_values
)

fare_median = manual_median(
    fare_values
)

fare_mode = manual_mode(
    fare_values
)

fare_variance = manual_sample_variance(
    fare_values
)

fare_std = manual_sample_std(
    fare_values
)

fare_minimum = min(
    fare_values
)

fare_maximum = max(
    fare_values
)


skew_n = fare_n

skew_sum = 0

for value in fare_values:

    standardized_value = (
        value - fare_mean
    ) / fare_std

    skew_sum += (
        standardized_value ** 3
    )

fare_skewness = (
    skew_n
    /
    ((skew_n - 1) * (skew_n - 2))
) * skew_sum


kurt_z_values = []

for value in fare_values:

    kurt_z_values.append(
        (value - fare_mean)
        /
        fare_std
    )

kurt_sum_4 = 0
kurt_sum_2 = 0

for z_value in kurt_z_values:

    kurt_sum_4 += z_value ** 4
    kurt_sum_2 += z_value ** 2

kurt_n = fare_n

fare_kurtosis = (
    (
        kurt_n * (kurt_n + 1)
        /
        (
            (kurt_n - 1)
            * (kurt_n - 2)
            * (kurt_n - 3)
        )
    )
    * kurt_sum_4
    -
    (
        3 * (kurt_n - 1) ** 2
        /
        (
            (kurt_n - 2)
            * (kurt_n - 3)
        )
    )
)


print("\nFARE STATISTICS")

print(
    "Mean:",
    round(fare_mean, 3)
)

print(
    "Median:",
    round(fare_median, 3)
)

print(
    "Mode:",
    fare_mode
)

print(
    "Variance:",
    round(fare_variance, 3)
)

print(
    "Standard Deviation:",
    round(fare_std, 3)
)

print(
    "Minimum:",
    round(fare_minimum, 3)
)

print(
    "Maximum:",
    round(fare_maximum, 3)
)

print(
    "Skewness:",
    round(fare_skewness, 3)
)

print(
    "Kurtosis:",
    round(fare_kurtosis, 3)
)


distance_values = distance.tolist()

distance_mean = manual_mean(
    distance_values
)

distance_median = manual_median(
    distance_values
)

distance_std = manual_sample_std(
    distance_values
)


print("\nDISTANCE STATISTICS")

print(
    "Mean:",
    round(distance_mean, 3)
)

print(
    "Median:",
    round(distance_median, 3)
)

print(
    "Standard Deviation:",
    round(distance_std, 3)
)


# Correlation

correlation_data = df[
    [
        "distance_km",
        "fare_amount"
    ]
].dropna()

x_values = correlation_data[
    "distance_km"
].tolist()

y_values = correlation_data[
    "fare_amount"
].tolist()

correlation = manual_correlation(
    x_values,
    y_values
)


print(
    "\nCorrelation between distance and fare:",
    round(correlation, 3)
)


# =============================================================================
# 13. PROBABILITY
# =============================================================================

print("\n" + "=" * 80)
print("13. PROBABILITY")
print("=" * 80)

total = len(df)

completed = (
    df["status"]
    .eq("Completed")
    .sum()
)

cancelled = (
    df["status"]
    .eq("Cancelled")
    .sum()
)

no_show = (
    df["status"]
    .eq("No-Show")
    .sum()
)


p_completed = (
    completed
    /
    total
)

p_cancelled = (
    cancelled
    /
    total
)

p_no_show = (
    no_show
    /
    total
)


print(
    "Total rides:",
    total
)

print(
    "\nCompleted rides:",
    completed
)

print(
    "P(Completed):",
    round(
        p_completed,
        4
    ),
    "=",
    round(
        p_completed * 100,
        2
    ),
    "%"
)


print(
    "\nCancelled rides:",
    cancelled
)

print(
    "P(Cancelled):",
    round(
        p_cancelled,
        4
    ),
    "=",
    round(
        p_cancelled * 100,
        2
    ),
    "%"
)


print(
    "\nNo-Show rides:",
    no_show
)

print(
    "P(No-Show):",
    round(
        p_no_show,
        4
    ),
    "=",
    round(
        p_no_show * 100,
        2
    ),
    "%"
)


# =============================================================================
# 14. SAMPLING AND INFERENCE
# =============================================================================

print("\n" + "=" * 80)
print("14. SAMPLING AND INFERENCE")
print("=" * 80)

sample_size = min(
    500,
    len(df)
)

sample = df.sample(
    n=sample_size,
    random_state=42
)


population_values = df[
    "fare_amount"
].values

population_mean = manual_mean(
    population_values.tolist()
)


sample_values = sample[
    "fare_amount"
].values

sample_mean = manual_mean(
    sample_values.tolist()
)

sample_n = len(
    sample_values
)

sample_std = manual_sample_std(
    sample_values.tolist()
)


print(
    "Population size:",
    len(df)
)

print(
    "Sample size:",
    len(sample)
)

print(
    "Population mean fare:",
    round(
        population_mean,
        3
    )
)

print(
    "Sample mean fare:",
    round(
        sample_mean,
        3
    )
)

print(
    "Sample standard deviation:",
    round(
        sample_std,
        3
    )
)


# Confidence Interval

confidence = 0.95

standard_error = (
    sample_std
    /
    np.sqrt(sample_size)
)

degrees_of_freedom = (
    sample_size - 1
)

alpha = 1 - confidence

t_critical = stats.t.ppf(
    1 - alpha / 2,
    degrees_of_freedom
)

margin_error = (
    t_critical
    *
    standard_error
)

lower_ci = (
    sample_mean
    -
    margin_error
)

upper_ci = (
    sample_mean
    +
    margin_error
)


print(
    "\n95% Confidence Interval for Sample Mean:"
)

print(
    "(",
    round(
        lower_ci,
        3
    ),
    ",",
    round(
        upper_ci,
        3
    ),
    ")"
)


# =============================================================================
# 15. HYPOTHESIS TESTING
# =============================================================================

print("\n" + "=" * 80)
print("15. HYPOTHESIS TESTING")
print("=" * 80)

weekday_fares = df[
    df["is_weekend"] == False
]["fare_amount"].dropna()

weekend_fares = df[
    df["is_weekend"] == True
]["fare_amount"].dropna()


print(
    "H0: Average weekday fare = Average weekend fare"
)

print(
    "H1: Average weekday fare != Average weekend fare"
)

print(
    "Significance level = 0.05"
)


weekday_values = weekday_fares.values
weekend_values = weekend_fares.values

n1 = len(
    weekday_values
)

n2 = len(
    weekend_values
)

weekday_mean = manual_mean(
    weekday_values.tolist()
)

weekend_mean = manual_mean(
    weekend_values.tolist()
)


weekday_variance = manual_sample_variance(
    weekday_values.tolist()
)

weekend_variance = manual_sample_variance(
    weekend_values.tolist()
)


difference_standard_error = np.sqrt(
    (
        weekday_variance / n1
    )
    +
    (
        weekend_variance / n2
    )
)


t_statistic = (
    weekday_mean
    -
    weekend_mean
) / difference_standard_error


df_numerator = (
    (
        weekday_variance / n1
        +
        weekend_variance / n2
    ) ** 2
)

df_denominator = (
    (
        (weekday_variance / n1) ** 2
        /
        (n1 - 1)
    )
    +
    (
        (weekend_variance / n2) ** 2
        /
        (n2 - 1)
    )
)

welch_degrees_of_freedom = (
    df_numerator
    /
    df_denominator
)


p_value = (
    2
    *
    stats.t.sf(
        abs(t_statistic),
        welch_degrees_of_freedom
    )
)


print(
    "\nWeekday average fare:",
    round(
        weekday_mean,
        3
    )
)

print(
    "Weekend average fare:",
    round(
        weekend_mean,
        3
    )
)

print(
    "T-statistic:",
    round(
        t_statistic,
        4
    )
)

print(
    "P-value:",
    round(
        p_value,
        4
    )
)


alpha = 0.05

if p_value < alpha:

    print(
        "\nDecision: Reject H0."
    )

    print(
        "There is statistically significant evidence of a difference."
    )

else:

    print(
        "\nDecision: Fail to reject H0."
    )

    print(
        "There is not sufficient statistical evidence of a difference."
    )


# =============================================================================
# 16. DATA VISUALIZATION
# =============================================================================

print("\n" + "=" * 80)
print("16. DATA VISUALIZATION")
print("=" * 80)


# Visualization 1

plt.figure(
    figsize=(8, 5)
)

sns.countplot(
    data=df,
    x="status"
)

plt.title(
    "Ride Status Distribution"
)

plt.xlabel(
    "Ride Status"
)

plt.ylabel(
    "Number of Rides"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()

plt.show()


# Visualization 2

plt.figure(
    figsize=(8, 5)
)

sns.countplot(
    data=df,
    x="payment_method"
)

plt.title(
    "Payment Method Distribution"
)

plt.xlabel(
    "Payment Method"
)

plt.ylabel(
    "Number of Rides"
)

plt.tight_layout()

plt.show()


# Visualization 3

plt.figure(
    figsize=(9, 5)
)

city_counts = (
    df["city"]
    .value_counts()
)

sns.barplot(
    x=city_counts.index,
    y=city_counts.values
)

plt.title(
    "Ride Count by City"
)

plt.xlabel(
    "City"
)

plt.ylabel(
    "Number of Rides"
)

plt.xticks(
    rotation=30
)

plt.tight_layout()

plt.show()


# Visualization 4

plt.figure(
    figsize=(8, 5)
)

sns.histplot(
    df["fare_amount"],
    bins=30,
    kde=True
)

plt.title(
    "Fare Distribution"
)

plt.xlabel(
    "Fare Amount"
)

plt.ylabel(
    "Frequency"
)

plt.tight_layout()

plt.show()


# Visualization 5

plt.figure(
    figsize=(8, 5)
)

sns.scatterplot(
    data=df,
    x="distance_km",
    y="fare_amount",
    alpha=0.4
)

plt.title(
    "Distance vs Fare"
)

plt.xlabel(
    "Distance (km)"
)

plt.ylabel(
    "Fare Amount"
)

plt.tight_layout()

plt.show()


# Visualization 6

hour_counts = (
    df["hour"]
    .value_counts()
    .sort_index()
)

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    hour_counts.index,
    hour_counts.values,
    marker="o"
)

plt.title(
    "Ride Demand by Hour"
)

plt.xlabel(
    "Hour of Day"
)

plt.ylabel(
    "Number of Rides"
)

plt.xticks(
    range(0, 24)
)

plt.grid(
    True
)

plt.tight_layout()

plt.show()


# Visualization 7

daily_demand = (
    df.groupby(
        df["pickup_time"].dt.date
    )
    .size()
)

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    daily_demand.index,
    daily_demand.values,
    marker="o"
)

plt.title(
    "Daily Ride Demand"
)

plt.xlabel(
    "Date"
)

plt.ylabel(
    "Number of Rides"
)

plt.xticks(
    rotation=45
)

plt.grid(
    True
)

plt.tight_layout()

plt.show()


# =============================================================================
# 17. WEB SCRAPING
# =============================================================================

print("\n" + "=" * 80)
print("17. WEB SCRAPING")
print("=" * 80)

try:

    url = "https://www.uber.com/us/en/ride/"

    response = requests.get(
        url,
        timeout=15,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    print(
        "HTTP Status Code:",
        response.status_code
    )

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    title = (
        soup.title.get_text(strip=True)
        if soup.title
        else
        "No title"
    )

    print(
        "Page Title:",
        title
    )

    headings = soup.find_all(
        ["h1", "h2", "h3"]
    )

    print("\nSample scraped headings:")

    for heading in headings[:10]:

        heading_text = heading.get_text(
            " ",
            strip=True
        )

        if heading_text:

            print(
                "-",
                heading_text
            )

    print(
        "\nRide-sharing web scraping completed successfully."
    )

except Exception as e:

    print(
        "Web scraping error:",
        e
    )


# =============================================================================
# 18. CRUD OPERATIONS
# =============================================================================

print("\n" + "=" * 80)
print("18. CRUD OPERATIONS")
print("=" * 80)

crud_df = df.copy()


# CREATE

print("\nCREATE - Adding a new ride")

new_ride = {

    "trip_id": "DEMO_NEW_001",
    "driver_id": "DRIVER_DEMO",
    "rider_id": "RIDER_DEMO",
    "city": "New York",

    "pickup_lat": 40.7128,
    "pickup_lng": -74.0060,

    "drop_lat": 40.7306,
    "drop_lng": -73.9352,

    "distance_km": 5.0,

    "fare_amount": 15.0,

    "status": "Completed",

    "payment_method": "Card",

    "pickup_time": pd.Timestamp(
        "2023-01-15 10:00:00"
    ),

    "drop_time": pd.Timestamp(
        "2023-01-15 10:20:00"
    )
}


crud_df = pd.concat(
    [
        crud_df,
        pd.DataFrame([new_ride])
    ],
    ignore_index=True
)

print(
    "New ride created."
)


# READ

print("\nREAD - Searching for the new ride")

read_result = crud_df[
    crud_df["trip_id"]
    ==
    "DEMO_NEW_001"
]

display(
    read_result
)


# UPDATE

print("\nUPDATE - Updating fare")

crud_df.loc[
    crud_df["trip_id"]
    ==
    "DEMO_NEW_001",
    "fare_amount"
] = 20.0

print(
    "Fare updated successfully."
)

display(
    crud_df[
        crud_df["trip_id"]
        ==
        "DEMO_NEW_001"
    ]
)


# DELETE

print("\nDELETE - Removing the demo ride")

crud_df = crud_df[
    crud_df["trip_id"]
    !=
    "DEMO_NEW_001"
]

print(
    "Demo ride deleted successfully."
)

print(
    "Ride still exists:",
    "DEMO_NEW_001"
    in
    crud_df["trip_id"].values
)


# =============================================================================
# 19. RECOMMENDATION SYSTEM
# =============================================================================

print("\n" + "=" * 80)
print("19. RIDE RECOMMENDATION SYSTEM")
print("=" * 80)


# ---------------------------------------------------------
# Recommendation Function
# ---------------------------------------------------------

def recommend_ride(
    city,
    distance,
    payment_method=None
):

    print("\n" + "-" * 60)
    print("RIDE RECOMMENDATION")
    print("-" * 60)

    print(
        "Requested City:",
        city
    )

    print(
        "Requested Distance:",
        distance,
        "km"
    )

    if payment_method:

        print(
            "Payment Method:",
            payment_method
        )


    # -----------------------------------------------------
    # Find rides from selected city
    # -----------------------------------------------------

    recommended_data = df[
        df["city"]
        .astype(str)
        .str.lower()
        ==
        city.lower()
    ].copy()


    # -----------------------------------------------------
    # Only completed rides
    # -----------------------------------------------------

    recommended_data = recommended_data[
        recommended_data["status"]
        .astype(str)
        .str.lower()
        ==
        "completed"
    ]


    # -----------------------------------------------------
    # Distance-based filtering
    # -----------------------------------------------------

    if distance <= 5:

        recommended_data = recommended_data[
            recommended_data["distance_km"] <= 5
        ]

        recommendation = "Short Ride"

    elif distance <= 10:

        recommended_data = recommended_data[
            (
                recommended_data["distance_km"] > 5
            )
            &
            (
                recommended_data["distance_km"] <= 10
            )
        ]

        recommendation = "Medium Ride"

    else:

        recommended_data = recommended_data[
            recommended_data["distance_km"] > 10
        ]

        recommendation = "Long Ride"


    # -----------------------------------------------------
    # Payment method filtering
    # -----------------------------------------------------

    if payment_method:

        payment_data = recommended_data[
            recommended_data["payment_method"]
            .astype(str)
            .str.lower()
            ==
            payment_method.lower()
        ]


        # Use payment-filtered data only if records exist

        if len(payment_data) > 0:

            recommended_data = payment_data


    # -----------------------------------------------------
    # Recommendation Result
    # -----------------------------------------------------

    print(
        "\nRecommended Ride Category:",
        recommendation
    )


    # -----------------------------------------------------
    # Calculate Average Fare
    # -----------------------------------------------------

    if len(recommended_data) > 0:

        average_recommended_fare = (
            recommended_data[
                "fare_amount"
            ].mean()
        )

        average_recommended_distance = (
            recommended_data[
                "distance_km"
            ].mean()
        )

        average_recommended_duration = (
            recommended_data[
                "duration_min"
            ].mean()
        )


        print(
            "Number of Matching Rides:",
            len(recommended_data)
        )

        print(
            "Average Fare:",
            round(
                average_recommended_fare,
                2
            )
        )

        print(
            "Average Distance:",
            round(
                average_recommended_distance,
                2
            ),
            "km"
        )

        print(
            "Average Duration:",
            round(
                average_recommended_duration,
                2
            ),
            "minutes"
        )


        # -------------------------------------------------
        # Display top recommendations
        # -------------------------------------------------

        top_rides = (
            recommended_data[
                [
                    "trip_id",
                    "city",
                    "distance_km",
                    "fare_amount",
                    "duration_min",
                    "payment_method"
                ]
            ]
            .sort_values(
                by="fare_amount"
            )
            .head(5)
        )


        print(
            "\nTop 5 Recommended Rides:"
        )

        display(
            top_rides
        )

    else:

        print(
            "\nNo matching ride found."
        )

        print(
            "Try another city, distance or payment method."
        )

    return recommended_data


# =============================================================================
# SAMPLE RECOMMENDATION
# =============================================================================

print(
    "\nExample Recommendation:"
)

available_cities = (
    df["city"]
    .dropna()
    .unique()
)


if len(available_cities) > 0:

    example_city = available_cities[0]

    recommended_rides = recommend_ride(
        city=example_city,
        distance=7,
        payment_method=None
    )


# =============================================================================
# 19.1 RECOMMENDATION SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("RECOMMENDATION SYSTEM SUMMARY")
print("=" * 80)

print(
    """
The recommendation system recommends rides based on:

1. City
2. Ride distance
3. Ride status
4. Payment method
5. Distance category

The system searches historical completed rides
and recommends similar rides to the user.
"""
)


# =============================================================================
# 20. MACHINE LEARNING - FARE PREDICTION
# =============================================================================

print("\n" + "=" * 80)
print("20. MACHINE LEARNING - FARE PREDICTION")
print("=" * 80)


# ---------------------------------------------------------
# Select features
# ---------------------------------------------------------

features = [
    "distance_km",
    "duration_min",
    "hour",
    "peak_numeric"
]

target = "fare_amount"


ml_df = df[
    features + [target]
].dropna()


X = ml_df[
    features
]

y = ml_df[
    target
]


print(
    "Features used:"
)

print(
    features
)

print(
    "\nTarget variable:",
    target
)

print(
    "\nRecords available for machine learning:",
    len(ml_df)
)


# ---------------------------------------------------------
# Train-Test Split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print(
    "\nTraining records:",
    len(X_train)
)

print(
    "Testing records:",
    len(X_test)
)


# ---------------------------------------------------------
# Linear Regression
# ---------------------------------------------------------

model = LinearRegression()

model.fit(
    X_train,
    y_train
)


# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------

predictions = model.predict(
    X_test
)


# =============================================================================
# MODEL EVALUATION
# =============================================================================

actual = y_test.values
predicted = predictions


# MAE

absolute_errors = np.abs(
    actual
    -
    predicted
)

mae = (
    np.sum(absolute_errors)
    /
    len(actual)
)


# MSE

squared_errors = (
    actual
    -
    predicted
) ** 2

mse = (
    np.sum(squared_errors)
    /
    len(actual)
)


# RMSE

rmse = np.sqrt(
    mse
)


# R2

actual_mean = (
    np.sum(actual)
    /
    len(actual)
)

ss_res = np.sum(
    (
        actual
        -
        predicted
    ) ** 2
)

ss_tot = np.sum(
    (
        actual
        -
        actual_mean
    ) ** 2
)

r2 = (
    1
    -
    (
        ss_res
        /
        ss_tot
    )
)


print(
    "\nMODEL PERFORMANCE"
)

print(
    "MAE:",
    round(
        mae,
        4
    )
)

print(
    "RMSE:",
    round(
        rmse,
        4
    )
)

print(
    "R² Score:",
    round(
        r2,
        4
    )
)


# ---------------------------------------------------------
# MODEL COEFFICIENTS
# ---------------------------------------------------------

print(
    "\nMODEL COEFFICIENTS"
)

coefficients = pd.DataFrame({

    "Feature":
        features,

    "Coefficient":
        model.coef_

})

display(
    coefficients
)

print(
    "\nIntercept:",
    round(
        model.intercept_,
        4
    )
)


# ---------------------------------------------------------
# Prediction Equation
# ---------------------------------------------------------

print(
    "\nPrediction Equation:"
)

print(
    f"Fare = {model.intercept_:.4f}"
    f" + ({model.coef_[0]:.4f} × Distance)"
    f" + ({model.coef_[1]:.4f} × Duration)"
    f" + ({model.coef_[2]:.4f} × Hour)"
    f" + ({model.coef_[3]:.4f} × Peak)"
)


# =============================================================================
# 21. ACTUAL VS PREDICTED FARE
# =============================================================================

print("\n" + "=" * 80)
print("21. ACTUAL VS PREDICTED FARE")
print("=" * 80)

prediction_results = pd.DataFrame({

    "Actual Fare":
        y_test.values,

    "Predicted Fare":
        predictions

})

display(
    prediction_results.head(10)
)


plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    y_test,
    predictions,
    alpha=0.5
)

minimum = min(
    y_test.min(),
    predictions.min()
)

maximum = max(
    y_test.max(),
    predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.title(
    "Actual vs Predicted Fare"
)

plt.xlabel(
    "Actual Fare"
)

plt.ylabel(
    "Predicted Fare"
)

plt.tight_layout()

plt.show()


# =============================================================================
# 22. SAVE OUTPUT FILES
# =============================================================================

print("\n" + "=" * 80)
print("22. SAVING PROJECT OUTPUTS")
print("=" * 80)


# Clean dataset

cleaned_file = (
    "/content/uber_cleaned_dataset.csv"
)

df.to_csv(
    cleaned_file,
    index=False
)


# ---------------------------------------------------------
# City Summary
# ---------------------------------------------------------

city_summary = (
    df.groupby("city")
    .agg(
        Ride_Count=(
            "trip_id",
            "count"
        ),

        Average_Fare=(
            "fare_amount",
            "mean"
        ),

        Average_Distance=(
            "distance_km",
            "mean"
        ),

        Average_Duration=(
            "duration_min",
            "mean"
        )
    )
    .round(2)
    .reset_index()
)

city_summary.to_csv(
    "/content/city_summary.csv",
    index=False
)


# ---------------------------------------------------------
# Statistical Summary
# ---------------------------------------------------------

statistical_summary = pd.DataFrame({

    "Metric": [

        "Total Rides",
        "Average Fare",
        "Median Fare",
        "Average Distance",
        "Average Duration",
        "Fare-Distance Correlation",
        "Cancellation Probability",
        "Completed Probability",
        "No-Show Probability",
        "Weekday Average Fare",
        "Weekend Average Fare",
        "Hypothesis Test P-Value",
        "ML MAE",
        "ML RMSE",
        "ML R2"

    ],

    "Value": [

        len(df),

        fare_mean,

        fare_median,

        distance_mean,

        (
            duration.sum()
            /
            duration.count()
        ),

        correlation,

        p_cancelled,

        p_completed,

        p_no_show,

        weekday_mean,

        weekend_mean,

        p_value,

        mae,

        rmse,

        r2

    ]

})


statistical_summary.to_csv(
    "/content/statistical_summary.csv",
    index=False
)


# ---------------------------------------------------------
# ML Prediction Results
# ---------------------------------------------------------

prediction_results.to_csv(
    "/content/fare_prediction_results.csv",
    index=False
)


print(
    "Saved:"
)

print(
    "1. uber_cleaned_dataset.csv"
)

print(
    "2. city_summary.csv"
)

print(
    "3. statistical_summary.csv"
)

print(
    "4. fare_prediction_results.csv"
)


# =============================================================================
# 23. FINAL PROJECT SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("                 FINAL PROJECT SUMMARY")
print("=" * 80)

print(
    "\nPROJECT: Ride-Sharing Analytics and Fare Prediction"
)

print(
    "\nDataset Records:",
    len(df)
)

print(
    "Dataset Columns:",
    len(df.columns)
)

print(
    "Number of Cities:",
    df["city"].nunique()
)

print(
    "Average Fare:",
    round(
        fare_mean,
        2
    )
)

print(
    "Average Distance:",
    round(
        distance_mean,
        2
    ),
    "km"
)

duration_mean = (
    duration.sum()
    /
    duration.count()
)

print(
    "Average Duration:",
    round(
        duration_mean,
        2
    ),
    "minutes"
)

print(
    "Completed Ride Probability:",
    round(
        p_completed * 100,
        2
    ),
    "%"
)

print(
    "Cancelled Ride Probability:",
    round(
        p_cancelled * 100,
        2
    ),
    "%"
)

print(
    "No-Show Probability:",
    round(
        p_no_show * 100,
        2
    ),
    "%"
)

print(
    "Distance-Fare Correlation:",
    round(
        correlation,
        3
    )
)

print(
    "Hypothesis Test P-value:",
    round(
        p_value,
        4
    )
)

print(
    "Machine Learning MAE:",
    round(
        mae,
        4
    )
)

print(
    "Machine Learning RMSE:",
    round(
        rmse,
        4
    )
)

print(
    "Machine Learning R²:",
    round(
        r2,
        4
    )
)


# =============================================================================
# CIPAT TOPICS COMPLETED
# =============================================================================

print("\n" + "=" * 80)
print("              CIPAT TOPICS COMPLETED")
print("=" * 80)

cipat_topics = [

    "1. Python Core Concepts",
    "2. Data Structures",
    "3. Python Libraries",
    "4. Custom Modules",
    "5. Exception Handling",
    "6. Web Scraping",
    "7. File Handling",
    "8. CRUD Operations",
    "9. NumPy",
    "10. SciPy",
    "11. Pandas",
    "12. Data Preprocessing",
    "13. Data Visualization",
    "14. Statistical Analysis",
    "15. Probability",
    "16. Sampling & Inference",
    "17. Hypothesis Testing",
    "18. Machine Learning",
    "19. Recommendation System"

]

for topic in cipat_topics:

    print(
        "✓",
        topic
    )


# =============================================================================
# PROJECT COMPLETED
# =============================================================================

print("\n" + "=" * 80)
print("             PROJECT EXECUTION COMPLETED")
print("=" * 80)

print(
    """
The complete ride-sharing analytics project has been executed.

The project includes:

✓ Data Loading
✓ Data Cleaning
✓ Python Concepts
✓ Data Structures
✓ NumPy
✓ Pandas
✓ SciPy
✓ Statistical Analysis
✓ Probability
✓ Sampling and Inference
✓ Hypothesis Testing
✓ Data Visualization
✓ Web Scraping
✓ CRUD Operations
✓ Recommendation System
✓ Machine Learning
✓ Fare Prediction
✓ Actual vs Predicted Analysis
✓ Output File Generation

The numerical results are generated directly
from the uploaded dataset.
"""
)

       RIDE-SHARING ANALYTICS AND FARE PREDICTION
All required Python libraries imported successfully.

2. FILE HANDLING - DATASET UPLOAD
Please upload your Uber Trips 50K CSV or ZIP file.
